# Episode Category Classification Demo

This notebook demonstrates a simple baseline approach for classifying podcast episodes into categories using their transcribed content. We'll use a TF-IDF + Logistic Regression pipeline to predict the primary category (category1) of an episode based on its combined speaker turns.

**Key Points:**
- Uses aggregated episode text (combined speaker turns)
- Focuses on primary category prediction (category1)
- Implements a lightweight TF-IDF + LogReg baseline
- Includes basic evaluation and example predictions

This serves as a proof-of-concept for more sophisticated approaches we'll develop later.

In [ ]:
# Import required packages
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import time

# Set random seed for reproducibility
np.random.seed(42)

## Sample Selection and Label Extraction

Here we'll:
1. Load episode data and select relevant columns
2. Keep only episodes with non-null text and category
3. Focus on top-K most frequent categories
4. Create feature matrix X and target vector y

In [ ]:
# Load episode data (assuming we have episode_df1 from previous notebook)
# Select rows with non-null text and category
df = episode_df1[['epTitle', 'combined_turnText', 'category1']].dropna()

# Keep only top 8 most frequent categories
top_k = 8
top_categories = df['category1'].value_counts().nlargest(top_k).index
df = df[df['category1'].isin(top_categories)]

# Create feature matrix X and target vector y
X = df['combined_turnText']
y = df['category1']

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y)

print("Dataset shape:", df.shape)
print("\nCategory distribution:")
for i, category in enumerate(le.classes_):
    count = (y == i).sum()
    print(f"{category}: {count} episodes ({count/len(y):.1%})")

## Text Vectorization (TF-IDF)

Convert the text data into numerical features using TF-IDF (Term Frequency-Inverse Document Frequency). This:
- Captures word importance through term frequency and document frequency
- Handles common words appropriately through IDF weighting
- Creates a sparse feature matrix suitable for classification

In [ ]:
# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=10000,  # Limit vocabulary size
    min_df=5,           # Ignore terms that appear in < 5 documents
    ngram_range=(1, 2), # Use unigrams and bigrams
    stop_words='english'
)

# Fit and transform the text data
X_tfidf = tfidf.fit_transform(X)

print("TF-IDF feature matrix shape:", X_tfidf.shape)
print("Number of features:", len(tfidf.get_feature_names_out()))
print("\nSample features:", list(tfidf.get_feature_names_out())[:10])

## Train/Validation Split

Split the data into training and validation sets, ensuring:
- Stratified sampling to maintain class distribution
- Random state set for reproducibility
- 80/20 train/validation split

In [ ]:
# Split data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_tfidf, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Training set shape:", X_train.shape)
print("Validation set shape:", X_val.shape)

# Verify class distribution is maintained
print("\nClass distribution in training set:")
for i, category in enumerate(le.classes_):
    count = (y_train == i).sum()
    print(f"{category}: {count/len(y_train):.1%}")

## Train Baseline Classifier

Train a Logistic Regression model on the TF-IDF features:
- Use balanced class weights to handle any remaining class imbalance
- Set solver to 'saga' for efficient handling of large sparse matrices
- Time the training process

In [ ]:
# Initialize and train the classifier
clf = LogisticRegression(
    solver='saga',
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

# Time the training process
start_time = time.time()
clf.fit(X_train, y_train)
training_time = time.time() - start_time

print(f"Training completed in {training_time:.2f} seconds")
print(f"Model convergence: {clf.n_iter_} iterations")

## Evaluation Metrics and Visualization

Evaluate the model's performance using:
1. Classification report (precision, recall, F1-score)
2. Confusion matrix visualization
3. Overall accuracy score

In [ ]:
# Make predictions on validation set
y_pred = clf.predict(X_val)

# Print classification report
print("Classification Report:")
print(classification_report(y_val, y_pred, target_names=le.classes_))

# Create confusion matrix
cm = confusion_matrix(y_val, y_pred)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Inspect Predictions on Example Episodes

Let's look at some specific examples to understand how the model performs in practice:
- Show correct and incorrect predictions
- Display prediction probabilities for top categories
- Examine patterns in misclassifications

In [ ]:
# Get prediction probabilities
y_proba = clf.predict_proba(X_val)

# Create a DataFrame with example predictions
example_df = pd.DataFrame({
    'Episode Title': df['epTitle'].iloc[X_val.indices],
    'Text': df['combined_turnText'].iloc[X_val.indices].apply(lambda x: x[:300] + '...'),
    'True Category': le.inverse_transform(y_val),
    'Predicted Category': le.inverse_transform(y_pred)
})

# Add top 3 prediction probabilities and their categories
for i in range(3):
    top_indices = np.argsort(y_proba, axis=1)[:, -(i+1)]
    example_df[f'Prob {i+1}'] = np.take_along_axis(y_proba, top_indices[:, None], axis=1)
    example_df[f'Category {i+1}'] = le.inverse_transform(top_indices)

# Show some interesting examples (both correct and incorrect predictions)
print("Correct Predictions:")
print(example_df[example_df['True Category'] == example_df['Predicted Category']].head(3))
print("\nIncorrect Predictions:")
print(example_df[example_df['True Category'] != example_df['Predicted Category']].head(3))

## Save Model and Vectorizer

Save the trained model components for later use:
- TF-IDF vectorizer
- Label encoder
- Trained classifier

This allows us to easily load and use the model for new predictions.

In [ ]:
# Save model components
joblib.dump(tfidf, 'tfidf.joblib')
joblib.dump(le, 'label_encoder.joblib')
joblib.dump(clf, 'classifier.joblib')

print("Model components saved successfully!")
print("Files created:")
print("- tfidf.joblib")
print("- label_encoder.joblib")
print("- classifier.joblib")